In [1]:
# Recommended Architecture for homepay.sg
# ====================================

"""
PRODUCTION SETUP:

1. Images: Google Cloud Storage bucket
   - Bucket: gs://homepay-prod-images/
   - Public read access
   - Lifecycle rules for cost optimization

2. Model: Google Cloud Run (or Kubernetes)
   - Container with model files only (no images)
   - Auto-scaling based on traffic
   - Multiple regions for low latency

3. CDN: Cloud CDN or Cloudflare
   - Cache images globally
   - Reduce costs and improve speed

4. API: RESTful endpoints
   - /search - semantic search
   - /similar - find similar images  
   - /recommendations - user-based recs

Architecture Flow:
Frontend → API (Cloud Run) → Returns image URLs → CDN → GCS bucket
"""

# Updated Flask app for production
from flask import Flask, request, jsonify
from inference import ModelServer
import os

app = Flask(__name__)

# Configuration
BUCKET_NAME = os.environ.get('BUCKET_NAME', 'homepay-prod-images')
CDN_BASE_URL = os.environ.get('CDN_BASE_URL', f'https://storage.googleapis.com/{BUCKET_NAME}/')

# Initialize model (no images needed in container)
model_server = ModelServer()

@app.route('/search', methods=['POST'])
def search():
    data = request.get_json()
    query = data.get('query', '')
    top_k = data.get('top_k', 20)
    
    try:
        # Get recommendations (returns image_ids and scores)
        results = model_server.search(query, top_k)
        
        # Convert to full response with URLs
        response = {
            'success': True,
            'query': query,
            'results': [
                {
                    'image_id': image_id,
                    'image_url': f"{CDN_BASE_URL}{image_id}",
                    'score': float(score),
                    'metadata': model_server.rec_system.data.get(image_id, {})
                }
                for image_id, score in results
            ]
        }
        
        return jsonify(response)
        
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/similar/<image_id>', methods=['GET'])
def get_similar(image_id):
    top_k = request.args.get('top_k', 20, type=int)
    user_id = request.args.get('user_id')
    
    try:
        results = model_server.get_similar(image_id, user_id=user_id, top_k=top_k)
        
        response = {
            'success': True,
            'reference_image': {
                'image_id': image_id,
                'image_url': f"{CDN_BASE_URL}{image_id}"
            },
            'similar_images': [
                {
                    'image_id': similar_id,
                    'image_url': f"{CDN_BASE_URL}{similar_id}",
                    'score': float(score)
                }
                for similar_id, score in results
            ]
        }
        
        return jsonify(response)
        
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=int(os.environ.get('PORT', 8080)))

ModuleNotFoundError: No module named 'flask'